<a href="https://colab.research.google.com/github/RoshanHelmy/FlyRank-Assignment1-/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
import numpy as np
import pandas as pd
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN found in Colab Secrets.")

# 2. Load the March 2026 warehouse data from Hugging Face


from huggingface_hub import hf_hub_download

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

performance_df = pd.read_parquet(performance_file)

print("Performance data loaded.")
print("Shape:", performance_df.shape)

# 3. Load the content table


content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content_df = pd.read_parquet(content_file)

print("Content data loaded.")
print("Shape:", content_df.shape)

# 4. Merge performance + content

baseline_df = performance_df.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)


# 5. Calculate CTR
baseline_df["ctr"] = np.where(
    baseline_df["gsc_impressions"] > 0,
    baseline_df["gsc_clicks"] / baseline_df["gsc_impressions"],
    0
)

# 6. Clean important numeric columns

baseline_df["gsc_impressions"] = pd.to_numeric(
    baseline_df["gsc_impressions"],
    errors="coerce"
).fillna(0)

baseline_df["gsc_avg_position"] = pd.to_numeric(
    baseline_df["gsc_avg_position"],
    errors="coerce"
)

baseline_df["ctr"] = pd.to_numeric(
    baseline_df["ctr"],
    errors="coerce"
).fillna(0)

print("\nFinal baseline dataframe:")
print("Shape:", baseline_df.shape)

display(
    baseline_df[
        [
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_avg_position",
            "gsc_clicks",
            "ctr"
        ]
    ].head()
)

HF_TOKEN found in Colab Secrets.


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Performance data loaded.
Shape: (9841378, 30)


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Content data loaded.
Shape: (519606, 26)

Final baseline dataframe:
Shape: (9841378, 55)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,3.350000,0,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0.000000,0,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,4.928000,1,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,4.000000,0,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,2.272727,0,0.000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



I chose Lane 2: Refresh / Content Opportunity Scoring.

My baseline rule will prioritize content pages that have meaningful search visibility but weaker search performance. The rule will use observable signals available at the decision moment, especially search impressions and average search position. Pages with more impressions and worse average position will receive a higher review score because they represent visible content with potential for improvement.

The rule is intended as a simple, transparent baseline for prioritizing which pages an editor or SEO team should review first.

### Reason code

The rule will use one reason code:

- `LOW_CTR_POSITION_OPPORTUNITY` — the page has search visibility but relatively weak search performance for its position.

### Action

The recommended action is:

- `REVIEW_FOR_REFRESH` — prioritize the page for human review to determine whether the content should be updated, expanded, or improved.

In [1]:
import os
import gc
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

# 1. Hugging Face authentication
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face Read token "
        "to Colab Secrets as HF_TOKEN."
    )

print("HF_TOKEN found in Colab Secrets.")

# 2. Download March 2026 performance parquet
REPO_ID = "FlyRank/internship-warehouse"

performance_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)
print("Performance file ready.")
# 3. Read ONLY the columns needed
performance_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

performance_df = pd.read_parquet(
    performance_path,
    columns=performance_columns
)

print("Performance data loaded.")
print("Shape:", performance_df.shape)
# 4. Keep only rows with usable GSC data
performance_df = performance_df[
    performance_df["gsc_data_available"].eq(True)
].copy()
# 5. Calculate CTR
performance_df["ctr"] = np.where(
    performance_df["gsc_impressions"] > 0,
    performance_df["gsc_clicks"] /
    performance_df["gsc_impressions"],
    0
)

# 6. Aggregate March daily data to PAGE level
page_df = (
    performance_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions_90d=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean")
    )
)

# Calculate page-level CTR
page_df["ctr"] = np.where(
    page_df["impressions_90d"] > 0,
    page_df["clicks"] / page_df["impressions_90d"],
    0
)

# 7. Clean numeric values

page_df["avg_position"] = (
    pd.to_numeric(
        page_df["avg_position"],
        errors="coerce"
    )
    .fillna(0)
)

page_df["impressions_90d"] = (
    pd.to_numeric(
        page_df["impressions_90d"],
        errors="coerce"
    )
    .fillna(0)
)

page_df["ctr"] = (
    pd.to_numeric(
        page_df["ctr"],
        errors="coerce"
    )
    .fillna(0)
)

# 8. Free the large dataframe
del performance_df
gc.collect()
# 9. Final dataframe
baseline_df = page_df.copy()

del page_df
gc.collect()

print("\nFinal page-level baseline dataframe:")
print("Shape:", baseline_df.shape)

display(
    baseline_df.head(10)
)

HF_TOKEN found in Colab Secrets.
Performance file ready.
Performance data loaded.
Shape: (9841378, 7)

Final page-level baseline dataframe:
Shape: (176738, 6)


,client_hash_id,content_hash_id,impressions_90d,clicks,avg_position,ctr
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.000000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.006042
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,0.000000
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.000000
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.000000
5,client_0797ff3a1fc9a6a5,content_12890868e4cdac06,1,0,19.000000,0.000000
6,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,39,1,9.168651,0.025641
7,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232,0,12.297523,0.000000
8,client_0797ff3a1fc9a6a5,content_1fea2f270f3c1350,4,0,4.166667,0.000000
9,client_0797ff3a1fc9a6a5,content_20346a450ede60c6,33,0,7.087500,0.000000


## Signal checks

### Signal 1: Search impressions

I use search impressions as a visibility signal. Higher impression volume means that a page is receiving more exposure in search, so a potential improvement may affect a meaningful amount of existing visibility.

**Verdict: CONFIRMED**

### Signal 2: Average search position

I use average search position as a performance signal. Pages in worse positions generally have lower CTR, which supports using position to identify pages that may have an opportunity for improvement.

**Verdict: CONFIRMED**

In [2]:

# Signal checks

# Signal 1: Impression buckets
baseline_df["impression_bucket"] = pd.cut(
    baseline_df["impressions_90d"],
    bins=[-1, 99, 499, np.inf],
    labels=["Low (0-99)", "Medium (100-499)", "High (500+)"]
)

impression_check = (
    baseline_df
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        mean_position=("avg_position", "mean")
    )
    .reset_index()
)

print("Signal 1 — Search impressions")
display(impression_check)


# Signal 2: Position buckets
baseline_df["position_bucket"] = pd.cut(
    baseline_df["avg_position"],
    bins=[-np.inf, 5, 15, np.inf],
    labels=["Top (<=5)", "Middle (5-15)", "Lower (>15)"]
)

position_check = (
    baseline_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        mean_impressions=("impressions_90d", "mean")
    )
    .reset_index()
)

print("\nSignal 2 — Average search position")
display(position_check)

Signal 1 — Search impressions


,impression_bucket,n,mean_ctr,mean_position
0,Low (0-99),75297,0.007259,18.101334
1,Medium (100-499),39517,0.002285,18.882510
2,High (500+),61924,0.002827,11.603319



Signal 2 — Average search position


,position_bucket,n,mean_ctr,mean_impressions
0,Top (<=5),44171,0.008785,2574.834643
1,Middle (5-15),74846,0.003961,1228.746266
2,Lower (>15),57721,0.002208,1298.623118


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

The score prioritizes pages with meaningful search visibility and weaker search position.

A higher score is assigned when:
- the page has more search impressions, and
- the page has a worse average search position.

The rule is a simple decision-support baseline, not a prediction of future performance.

Each ranked page receives one reason code, `LOW_CTR_POSITION_OPPORTUNITY`, and the action label `REVIEW_FOR_REFRESH`.

In [3]:

# Section 2 — Build the ranked baseline queue

# Work only with the columns needed for the rule
queue = baseline_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
].copy()

# Avoid extreme effects from very large impression values
# by using log-transformed impressions.
queue["impression_score"] = np.log1p(
    queue["impressions_90d"]
)

# Normalize position into a simple opportunity signal.
# Higher values = worse position = greater opportunity.
queue["position_score"] = np.clip(
    queue["avg_position"],
    0,
    50
)

# Final baseline score
queue["score"] = (
    queue["impression_score"] *
    queue["position_score"]
)

# One reason code
queue["reason_code"] = "LOW_CTR_POSITION_OPPORTUNITY"

# Action label
queue["action"] = np.where(
    queue["score"] > 0,
    "REVIEW_FOR_REFRESH",
    "NO_ACTION"
)

# Rank highest score first
queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Reorder columns
queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(
    output_path,
    index=False
)

print("Baseline queue created successfully.")
print("Rows:", len(queue))
print("Saved to:", output_path)

display(queue.head(10))

Baseline queue created successfully.
Rows: 176738
Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions_90d,avg_position,ctr
0,1,client_23a62021009f63c4,content_3f9e8f387f3fe7e7,531.835729,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,53620,48.838436,0.000466
1,2,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,529.083668,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,39405,54.384283,0.000076
2,3,client_23a62021009f63c4,content_2de9a39d3482a269,518.148362,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,31664,53.143951,0.000189
3,4,client_23a62021009f63c4,content_73aa61dcedebbf30,516.019247,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,80124,45.700431,0.000112
4,5,client_23a62021009f63c4,content_1ff3c48911f11e70,511.432301,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,27684,55.147128,0.000072
5,6,client_23a62021009f63c4,content_c367b0ca57f3559b,510.858919,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,62928,46.232571,0.000016
6,7,client_23a62021009f63c4,content_96e6613b42b52c42,509.526752,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,63819,46.053412,0.000078
7,8,client_23a62021009f63c4,content_392abd14d5a146ea,501.673382,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,23536,49.836776,0.000255
8,9,client_23a62021009f63c4,content_0cf7684dbe872d01,499.941434,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,39331,47.254365,0.000407
9,10,client_23a62021009f63c4,content_295e883e0e86ca3c,499.803337,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,21939,51.573254,0.000000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



The following review checks the highest-ranked pages manually. The ranking is only a prioritization baseline, so each recommendation should be treated as decision support rather than an automatic instruction to refresh a page.

For each page, I consider:
- the recommended action,
- the reason for its ranking,
- how confident the signal appears, and
- what could make the recommendation wrong.

In [4]:

# Section 3 — Top-20 review

top20 = queue.head(20).copy()

# Confidence note based on the strength of the baseline signals
def confidence_note(row):
    if row["impressions_90d"] >= 500 and row["avg_position"] > 15:
        return "Higher confidence: meaningful visibility and weak position."
    elif row["impressions_90d"] >= 100 and row["avg_position"] > 10:
        return "Moderate confidence: useful visibility and weaker position."
    else:
        return "Lower confidence: one or more signals are relatively weak."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The page may not need a refresh if the low position reflects "
    "search intent, SERP competition, or another factor that content "
    "changes would not address."
)

review_columns = [
    "rank",
    "content_hash_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_3f9e8f387f3fe7e7,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
1,2,content_6aa54d6bbdbf6f24,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
2,3,content_2de9a39d3482a269,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
3,4,content_73aa61dcedebbf30,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
4,5,content_1ff3c48911f11e70,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
5,6,content_c367b0ca57f3559b,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
6,7,content_96e6613b42b52c42,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
7,8,content_392abd14d5a146ea,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
8,9,content_0cf7684dbe872d01,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...
9,10,content_295e883e0e86ca3c,REVIEW_FOR_REFRESH,LOW_CTR_POSITION_OPPORTUNITY,Higher confidence: meaningful visibility and w...,The page may not need a refresh if the low pos...


The top-20 review shows that the rule is useful for prioritization, but the recommendations are not automatic refresh decisions. A page can have high visibility and a weak position for reasons unrelated to content quality, such as competition or search intent. Human review is therefore required before taking action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some high-ranked pages may be weak recommendations because the rule only uses search visibility and position. High impressions do not necessarily mean that a content refresh will improve performance, and average position can be affected by competition or search intent.

These cases should therefore be reviewed by an editor or SEO specialist before action is taken.

### Leakage check

The baseline uses only signals available in the March 2026 performance data at the decision point: impressions, clicks/CTR, and average search position.

It does not use the future outcome label, trend direction, trend percentage, product decision flags, or future-window measurements.

Therefore, the baseline does not intentionally use label-derived or future information.

In [5]:

# Section 4 — Leakage check


# Features actually used by the baseline rule
used_features = [
    "impressions_90d",
    "avg_position"
]

# Explicitly check that common outcome/label fields
# are not present in the queue.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "needs_ctr_fix",
    "product_flag"
]

present_leakage_fields = [
    col for col in leakage_fields
    if col in queue.columns
]

print("Features used by baseline:")
print(used_features)

print("\nPotential leakage fields present in queue:")
print(present_leakage_fields)

if len(present_leakage_fields) == 0:
    print("\nLEAKAGE CHECK: PASSED")
    print("No label-derived or product decision fields are in the ranked queue.")
else:
    print("\nLEAKAGE CHECK: REVIEW REQUIRED")

Features used by baseline:
['impressions_90d', 'avg_position']

Potential leakage fields present in queue:
[]

LEAKAGE CHECK: PASSED
No label-derived or product decision fields are in the ranked queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.